# Lecture 3 — The 1D Quantum Transverse-Field Ising Model

---

## Overview

The Transverse-Field Ising Model (TFIM) is the simplest quantum model that has a phase transition. It is exactly solvable, connects directly to the 2D classical Ising model via a quantum-to-classical mapping, and serves as the paradigmatic example of a quantum phase transition. Everything we learn about it generalises to more complex quantum critical systems.

In this lecture we make the conceptual leap from classical to quantum spins, build the TFIM Hamiltonian explicitly from Pauli matrices and tensor products, and understand the two competing phases and the quantum-to-classical mapping. The spectrum and finite-size diagnostics are developed in lectures 05–06; the phase transition itself is quantified in lecture 04.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. From Classical Spins to Quantum Operators

A classical Ising spin $s_i \in \{+1, -1\}$ is replaced by a quantum spin-$1/2$ operator $\hat{\vec{S}}_i = (\hat{S}_i^x, \hat{S}_i^y, \hat{S}_i^z)$. Each component acts on a two-dimensional local Hilbert space spanned by $|\uparrow\rangle$ and $|\downarrow\rangle$. In matrix form, using the Pauli matrices:

$$\hat{\sigma}^x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad \hat{\sigma}^y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad \hat{\sigma}^z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

The defining property of quantum spin operators is the commutation relation $[\hat{\sigma}^x, \hat{\sigma}^y] = 2i\hat{\sigma}^z$ and cyclic permutations. This non-commutativity means the three components cannot be simultaneously specified — measuring $\sigma^z$ disturbs $\sigma^x$ and $\sigma^y$.

For an $N$-site chain, the total Hilbert space is $\mathcal{H} = (\mathbb{C}^2)^{\otimes N} \cong \mathbb{C}^{2^N}$ — exponential in system size.

---

In [ ]:
# Define the Pauli matrices
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

# Verify the commutation relation [sigma_x, sigma_y] = 2i * sigma_z
commutator = sx @ sy - sy @ sx
print("[σx, σy] =")
print(commutator)
print("\n2i·σz =")
print(2j * sz)
print("\nMatch:", np.allclose(commutator, 2j * sz))

## 2. The Hamiltonian

The 1D TFIM on a chain of $N$ sites is:

$$\hat{H} = -J \sum_{i=1}^{N-1} \hat{\sigma}^z_i \hat{\sigma}^z_{i+1} - \Gamma \sum_{i=1}^{N} \hat{\sigma}^x_i$$

There are two competing terms:
- The **Ising coupling** $-J \hat{\sigma}^z_i \hat{\sigma}^z_{i+1}$: favours aligned spins in the $z$-direction.
- The **transverse field** $-\Gamma \hat{\sigma}^x_i$: causes spins to tunnel between $|\uparrow\rangle$ and $|\downarrow\rangle$.

The competition between these two terms drives the quantum phase transition at $\Gamma_c = J$.

---

## 3. The Two Limits

**Limit 1: $\Gamma = 0$ (pure Ising)** — The ground states are $|\uparrow\uparrow\cdots\uparrow\rangle$ and $|\downarrow\downarrow\cdots\downarrow\rangle$, doubly degenerate with energy $-J(N-1)$. The system is in the **ordered phase**.

**Limit 2: $J = 0$ (pure transverse field)** — The unique ground state is $|+\rangle^{\otimes N} = \left((|\uparrow\rangle + |\downarrow\rangle)/\sqrt{2}\right)^{\otimes N}$, with energy $-\Gamma N$. This is the **disordered phase**.

---

In [ ]:
def build_tfim(N, J, Gamma):
    """Build the TFIM Hamiltonian as a sparse matrix for an open chain of N sites."""
    dim = 2**N
    H = sp.lil_matrix((dim, dim), dtype=float)

    # Ising coupling: -J * sigma_z_i * sigma_z_{i+1}
    for i in range(N - 1):
        for state in range(dim):
            bit_i = (state >> i) & 1
            bit_j = (state >> (i + 1)) & 1
            # sigma_z eigenvalue: +1 if bit=1 (up), -1 if bit=0 (down)
            sz_i = 1 if bit_i else -1
            sz_j = 1 if bit_j else -1
            H[state, state] -= J * sz_i * sz_j

    # Transverse field: -Gamma * sigma_x_i
    for i in range(N):
        for state in range(dim):
            # sigma_x flips bit i
            flipped = state ^ (1 << i)
            H[state, flipped] -= Gamma

    return H.tocsr()


N = 6

# Limit 1: pure Ising (Gamma = 0)
H_ising = build_tfim(N, J=1.0, Gamma=0.0)
evals_ising = np.sort(np.linalg.eigvalsh(H_ising.toarray()))
print(f"Γ=0 (pure Ising), N={N}:")
print(f"  Ground state energy: {evals_ising[0]:.4f}  (expected: {-(N-1):.4f})")
print(f"  First two levels: {evals_ising[0]:.6f}, {evals_ising[1]:.6f}")
print(f"  Gap (should be ~0 for ordered phase): {evals_ising[1]-evals_ising[0]:.2e}")

# Limit 2: pure transverse field (J = 0)
H_field = build_tfim(N, J=0.0, Gamma=1.0)
evals_field = np.sort(np.linalg.eigvalsh(H_field.toarray()))
print(f"\nJ=0 (pure transverse field), N={N}:")
print(f"  Ground state energy: {evals_field[0]:.4f}  (expected: {-N:.4f})")
print(f"  Gap (should be finite): {evals_field[1]-evals_field[0]:.4f}")

## 4. Building the Hamiltonian from Pauli Matrices

For an operator acting on site $i$ of an $N$-site chain, we use the tensor product:

$$\hat{\sigma}^z_i = \underbrace{I \otimes \cdots \otimes I}_{i-1} \otimes \sigma^z \otimes \underbrace{I \otimes \cdots \otimes I}_{N-i}$$

This gives a $2^N \times 2^N$ matrix that acts as $\sigma^z$ on site $i$ and as the identity everywhere else. The full Hamiltonian is a sum of all such terms, built as a sparse matrix.

Below is a second implementation using explicit Kronecker products — more transparent but equivalent to the bit-manipulation approach above.

---

In [ ]:
def kron_site_op(op, i, N):
    """Embed a single-site operator `op` at site i in an N-site chain."""
    matrices = [sp.eye(2, format='csr')] * N
    matrices[i] = sp.csr_matrix(op)
    result = matrices[0]
    for m in matrices[1:]:
        result = sp.kron(result, m, format='csr')
    return result


def build_tfim_kron(N, J, Gamma):
    """Build TFIM Hamiltonian via explicit Kronecker products."""
    dim = 2**N
    H = sp.csr_matrix((dim, dim), dtype=float)

    for i in range(N - 1):
        Sz_i = kron_site_op(sz.real, i, N)
        Sz_j = kron_site_op(sz.real, i + 1, N)
        H -= J * Sz_i.dot(Sz_j)

    for i in range(N):
        Sx_i = kron_site_op(sx.real, i, N)
        H -= Gamma * Sx_i

    return H


# Verify both constructions agree
N = 4
H1 = build_tfim(N, J=1.0, Gamma=0.7)
H2 = build_tfim_kron(N, J=1.0, Gamma=0.7)
print(f"Max difference between the two constructions: {np.max(np.abs((H1 - H2).toarray())):.2e}")
print("Both constructions agree." if np.allclose(H1.toarray(), H2.toarray()) else "MISMATCH!")

## 5. What Makes This Quantum?

A classical phase transition is driven by thermal energy $k_BT$ competing with interaction energy $J$. Set $T = 0$ and the classical system freezes — no transition as a function of other parameters.

A quantum phase transition occurs at $T = 0$ as a function of a coupling constant in the Hamiltonian. The key difference:

| | Classical (Ising, $T$ sweep) | Quantum (TFIM, $\Gamma$ sweep) |
|---|---|---|
| Control parameter | Temperature $T$ | Transverse field $\Gamma$ |
| Fluctuations | Thermal | Quantum (zero-point) |
| Occurs at | $T = T_c > 0$ | $\Gamma = \Gamma_c$, $T = 0$ |
| Order parameter | $\langle \sigma^z \rangle$ (thermal avg) | $\langle \hat\sigma^z \rangle$ (ground state) |

Despite these differences, the mathematical structure is identical. The FSS tools from lecture 02 apply directly — we use them in lecture 04.

---

In [ ]:
from scipy.sparse.linalg import eigsh

# Show ground state energy and order parameter across the phase diagram
N = 10
gammas = np.linspace(0.1, 2.5, 50)
energies = []
mag_sq = []

# Build sigma_z total operator
Mz = sum(kron_site_op(sz.real, i, N) for i in range(N)) / N

for g in gammas:
    H = build_tfim(N, J=1.0, Gamma=g)
    evals, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]
    energies.append(evals[0] / N)
    m2 = psi0 @ (Mz @ Mz) @ psi0
    mag_sq.append(float(m2.real))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(gammas, energies, 'b-')
ax1.axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c = J$')
ax1.set_xlabel(r'$\Gamma / J$')
ax1.set_ylabel(r'$E_0 / N$')
ax1.set_title('Ground state energy per site')
ax1.legend()

ax2.plot(gammas, mag_sq, 'g-')
ax2.axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c = J$')
ax2.set_xlabel(r'$\Gamma / J$')
ax2.set_ylabel(r'$\langle m^2 \rangle$')
ax2.set_title(r'Order parameter $\langle m^2 \rangle$ vs $\Gamma/J$')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. The Quantum-to-Classical Mapping

There is a profound connection between the 1D quantum TFIM and the 2D classical Ising model. Using the Suzuki-Trotter decomposition, the quantum partition function $Z = \text{Tr}\, e^{-\beta \hat{H}}$ can be written as a sum over classical spin configurations on a 2D lattice — one spatial dimension plus one imaginary time dimension.

At $T = 0$ ($\beta \to \infty$), this imaginary time direction becomes infinite, and the effective 2D classical model is the 2D Ising model. The consequence:

> **The critical exponents of the 1D TFIM are the same as those of the 2D classical Ising model:** $\beta = 1/8$, $\nu = 1$, $\gamma = 7/4$.

The exponents we measured in lecture 02 apply directly to the quantum phase transition in lecture 04. This mapping — between a $d$-dimensional quantum system and a $(d+z)$-dimensional classical system — is one of the deepest ideas in condensed matter theory.

---

In [ ]:
# Verify: at the critical point Gamma=J=1, check the known exact ground state energy
# For open boundary conditions, E0/N -> -2J/pi as N -> infinity
exact_limit = -2 / np.pi

print(f"{'N':>5}  {'E0/N (ED)':>12}  {'exact limit':>12}  {'error':>10}")
print("-" * 45)
for N in [4, 6, 8, 10, 12, 14]:
    H = build_tfim(N, J=1.0, Gamma=1.0)
    evals = eigsh(H, k=1, which='SA', return_eigenvectors=False)
    e0_per_site = evals[0] / N
    print(f"{N:>5}  {e0_per_site:>12.6f}  {exact_limit:>12.6f}  {abs(e0_per_site - exact_limit):>10.2e}")

print(f"\nThermodynamic limit: E0/N → -2/π = {exact_limit:.6f}")